In [0]:
# Célula 1 — criar schema silver
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
print("Schema silver criado!")

Schema silver criado!


In [0]:
# Célula 2 — silver: orders
from pyspark.sql.functions import current_timestamp, lit, col, to_timestamp

df_raw = spark.table("workspace.raw.olist_orders_dataset")

df_silver = df_raw \
    .dropDuplicates(["order_id"]) \
    .filter(col("order_status").isNotNull()) \
    .withColumn("order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("olist_orders_dataset.csv"))

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.olist_orders")

print(f"Silver orders: {df_silver.count()} linhas salvas!")

Silver orders: 99441 linhas salvas!


In [0]:
# Célula 3 — silver: customers, payments, sellers
from pyspark.sql.functions import current_timestamp, lit, col

# customers
df = spark.table("workspace.raw.olist_customers_dataset") \
    .dropDuplicates(["customer_id"]) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("olist_customers_dataset.csv"))
df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.olist_customers")
print(f"Silver customers: {df.count()} linhas")

# payments
df = spark.table("workspace.raw.olist_order_payments_dataset") \
    .filter(col("payment_value") > 0) \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("olist_order_payments_dataset.csv"))
df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.olist_payments")
print(f"Silver payments: {df.count()} linhas")

# sellers
df = spark.table("workspace.raw.olist_sellers_dataset") \
    .dropDuplicates(["seller_id"]) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("olist_sellers_dataset.csv"))
df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.olist_sellers")
print(f"Silver sellers: {df.count()} linhas")

print("\nCustomers, payments e sellers salvos!")

Silver customers: 99441 linhas
Silver payments: 103877 linhas
Silver sellers: 3095 linhas

Customers, payments e sellers salvos!


In [0]:
# Célula 4 — silver: reviews (try_cast)
from pyspark.sql.functions import current_timestamp, lit, col, expr

df = spark.table("workspace.raw.olist_order_reviews_dataset") \
    .dropDuplicates(["review_id"]) \
    .filter(col("review_score").isNotNull()) \
    .withColumn("review_score", expr("try_cast(review_score as integer)")) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("olist_order_reviews_dataset.csv"))

df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.olist_reviews")
print(f"Silver reviews: {df.count()} linhas")
print("Camada Silver Finalizada")

Silver reviews: 100643 linhas
Camada Silver Finalizada
